# 06 Eval with LLM-as-Judge

Unified evaluation and comparison notebook. Loads existing prediction JSONs from all runs (no GPU needed):
- Computes EM metrics for baseline, D1, D2, D3
- Runs LLM-as-judge (GPT-4o-mini) for semantic equivalence on answerable predictions
- Combined comparison table: EM + LLM-judge metrics
- Per-unanswerable-type abstention recall breakdown
- Training curves comparison across D1, D2, D3

In [ ]:
%pip install -q openai matplotlib

In [ ]:
from pathlib import Path
import json
import sys

from google.colab import drive
drive.mount('/content/drive')

ROOT = Path("/content/drive/MyDrive/abstention-data")

if not (ROOT / "src").exists():
    raise FileNotFoundError(
        f"Invalid ROOT: {ROOT}. Make sure repo is in Drive at /content/drive/MyDrive/abstention-data"
    )

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# Prediction file paths
BASELINE_DIR = ROOT / "outputs" / "notebooks" / "baseline"
LORA1_DIR = ROOT / "outputs" / "notebooks" / "lora_dataset1" / "eval"
LORA2_DIR = ROOT / "outputs" / "notebooks" / "lora_dataset2" / "eval"
LORA3_DIR = ROOT / "outputs" / "notebooks" / "lora_dataset3" / "eval"
OUT_DIR = ROOT / "outputs" / "notebooks" / "comparison"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)

## Load Predictions from All Runs

In [ ]:
from abstention_pipeline.evaluation import load_predictions
from abstention_pipeline.metrics import compute_metrics, compute_metrics_with_judge
from abstention_pipeline.data import load_jsonl

# Auto-discover all runs: baseline + any lora_* directories
OUTPUTS_DIR = ROOT / "outputs" / "notebooks"

pred_paths = {}

# Baseline
baseline_d4 = OUTPUTS_DIR / "baseline" / "dataset4.json"
if baseline_d4.exists():
    pred_paths["baseline"] = baseline_d4

# Discover lora runs
for lora_dir in sorted(OUTPUTS_DIR.glob("lora_*")):
    eval_dir = lora_dir / "eval"
    # Try predictions-only first, fall back to full report
    pred_file = eval_dir / "dataset4_predictions.json"
    if not pred_file.exists():
        pred_file = eval_dir / "dataset4.json"
    if pred_file.exists():
        # Name: lora_dataset1 -> D1, lora_dataset3c -> D3c, etc.
        run_name = lora_dir.name.replace("lora_dataset", "D")
        pred_paths[run_name] = pred_file

all_preds = {}
for name, path in pred_paths.items():
    all_preds[name] = load_predictions(path)
    print(f"Loaded {name}: {len(all_preds[name])} predictions from {path.name}")

# Backfill 'input' field for older predictions
dataset4_rows = load_jsonl(ROOT / "data" / "dataset4.jsonl")
id_to_input = {row["id"]: row["input"] for row in dataset4_rows}

for name, preds in all_preds.items():
    missing = sum(1 for r in preds if "input" not in r)
    if missing:
        for r in preds:
            if "input" not in r:
                r["input"] = id_to_input.get(r["id"], r["id"])
        print(f"  Backfilled {missing} 'input' fields for {name}")

print(f"\nDiscovered {len(all_preds)} runs: {list(all_preds.keys())}")

## EM Metrics (All Runs)

In [ ]:
em_results = {}
for name, preds in all_preds.items():
    em_results[name] = compute_metrics(preds)

# Print comparison table
metrics_to_show = [
    "overall_exact_match",
    "answerable_exact_match",
    "abstain_precision",
    "abstain_recall",
    "abstain_f1",
    "pred_abstain_rate",
    "gold_abstain_rate",
]

print(f"{'Metric':<30}", end="")
for name in em_results:
    print(f"{name:>15}", end="")
print()
print("-" * (30 + 15 * len(em_results)))

for metric in metrics_to_show:
    print(f"{metric:<30}", end="")
    for name in em_results:
        val = em_results[name].get(metric, "N/A")
        if isinstance(val, float):
            print(f"{val:>15.4f}", end="")
        else:
            print(f"{str(val):>15}", end="")
    print()

## LLM-as-Judge Evaluation

In [ ]:
from abstention_pipeline.llm_judge import judge_batch

# Set your OpenAI API key here
OPENAI_API_KEY = "YOUR_KEY_HERE"

# Run LLM judge on all prediction sets
judged_preds = {}
for name, preds in all_preds.items():
    print(f"\nJudging {name} ({len(preds)} predictions)...")
    judged = judge_batch(
        records=preds,
        api_key=OPENAI_API_KEY,
        model="gpt-5.2",
        requests_per_minute=2000,
    )
    judged_preds[name] = judged
    print(f"  Done. Example: {judged[0].get('llm_judge_correct', 'N/A')}")

## Combined Comparison: EM + LLM Judge

In [ ]:
# Compute combined metrics with judge
judge_results = {}
for name, preds in judged_preds.items():
    judge_results[name] = compute_metrics_with_judge(preds)

# Combined table: EM + LLM judge metrics
combined_metrics = [
    "overall_exact_match",
    "overall_llm_accuracy",
    "answerable_exact_match",
    "answerable_llm_accuracy",
    "abstain_precision",
    "abstain_recall",
    "abstain_f1",
    "pred_abstain_rate",
    "gold_abstain_rate",
]

print(f"{'Metric':<30}", end="")
for name in judge_results:
    print(f"{name:>15}", end="")
print()
print("-" * (30 + 15 * len(judge_results)))

for metric in combined_metrics:
    print(f"{metric:<30}", end="")
    for name in judge_results:
        val = judge_results[name].get(metric, "N/A")
        if isinstance(val, float):
            print(f"{val:>15.4f}", end="")
        else:
            print(f"{str(val):>15}", end="")
    print()

## Metrics Comparison Bar Charts

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

run_names = list(judge_results.keys())
n_runs = len(run_names)
colors = ["#4C72B0", "#55A868", "#C44E52", "#8172B2"][:n_runs]

# --- Chart 1: Abstention metrics ---
abstain_metrics = ["abstain_precision", "abstain_recall", "abstain_f1"]
abstain_labels = ["Precision", "Recall", "F1"]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, (metric, label) in enumerate(zip(abstain_metrics, abstain_labels)):
    vals = [judge_results[r].get(metric, 0) for r in run_names]
    bars = axes[i].bar(run_names, vals, color=colors)
    axes[i].set_title(f"Abstention {label}")
    axes[i].set_ylim(0, 1.05)
    axes[i].grid(axis="y", alpha=0.3)
    for bar, v in zip(bars, vals):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                     f"{v:.3f}", ha="center", va="bottom", fontsize=9)

fig.suptitle("Abstention Metrics by Run", fontsize=14)
fig.tight_layout()
fig.savefig(str(OUT_DIR / "abstention_metrics_bars.png"), dpi=150, bbox_inches="tight")
plt.show()

# --- Chart 2: Answerable accuracy (EM vs Judge) ---
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(n_runs)
width = 0.35

em_vals = [judge_results[r].get("answerable_exact_match", 0) for r in run_names]
judge_vals = [judge_results[r].get("answerable_llm_accuracy", 0) for r in run_names]

bars1 = ax.bar(x - width/2, em_vals, width, label="Exact Match", color="#4C72B0")
bars2 = ax.bar(x + width/2, judge_vals, width, label="LLM Judge", color="#55A868")

ax.set_ylabel("Accuracy")
ax.set_title("Answerable Accuracy: Exact Match vs LLM Judge")
ax.set_xticks(x)
ax.set_xticklabels(run_names)
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(axis="y", alpha=0.3)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=9)

fig.tight_layout()
fig.savefig(str(OUT_DIR / "answerable_accuracy_bars.png"), dpi=150, bbox_inches="tight")
plt.show()

# --- Chart 3: Pred abstain rate vs gold ---
fig, ax = plt.subplots(figsize=(8, 5))
pred_rates = [judge_results[r].get("pred_abstain_rate", 0) for r in run_names]
gold_rate = judge_results[run_names[0]].get("gold_abstain_rate", 0.5)

ax.bar(run_names, pred_rates, color=colors)
ax.axhline(y=gold_rate, color="red", linestyle="--", linewidth=2, label=f"Gold rate ({gold_rate:.1%})")
ax.set_ylabel("Abstain Rate")
ax.set_title("Predicted Abstain Rate vs Gold")
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(axis="y", alpha=0.3)

for i, v in enumerate(pred_rates):
    ax.text(i, v + 0.02, f"{v:.3f}", ha="center", va="bottom", fontsize=9)

fig.tight_layout()
fig.savefig(str(OUT_DIR / "abstain_rate_bars.png"), dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved charts to {OUT_DIR}")

## Per-Unanswerable-Type Abstention Recall

In [ ]:
# Per-unanswerable-type breakdown
utype_data = {}  # {run_name: {utype: recall}}
for name, result in judge_results.items():
    per_type = result.get("per_unanswerable_type", {})
    for utype, info in per_type.items():
        utype_data.setdefault(utype, {})[name] = info.get("abstain_recall", 0)

if utype_data:
    all_types = sorted(utype_data.keys())
    run_names = list(judge_results.keys())

    print(f"{'Unanswerable Type':<30}", end="")
    for name in run_names:
        print(f"{name:>15}", end="")
    print()
    print("-" * (30 + 15 * len(run_names)))

    for utype in all_types:
        print(f"{utype:<30}", end="")
        for name in run_names:
            val = utype_data.get(utype, {}).get(name, "N/A")
            if isinstance(val, float):
                print(f"{val:>15.4f}", end="")
            else:
                print(f"{str(val):>15}", end="")
        print()
else:
    print("No per-type data available.")

## Training Curves Comparison (D1, D2, D3)

In [ ]:
import matplotlib.pyplot as plt
from abstention_pipeline.visualization import load_trainer_state, extract_training_curves

# Auto-discover training dirs
all_curves = {}
for lora_dir in sorted(OUTPUTS_DIR.glob("lora_*")):
    state_path = lora_dir / "training" / "trainer_state.json"
    if state_path.exists():
        run_name = lora_dir.name.replace("lora_dataset", "D")
        state = load_trainer_state(state_path)
        all_curves[run_name] = extract_training_curves(state)
        print(f"Loaded {run_name}: {len(all_curves[run_name]['step'])} training steps")

if all_curves:
    fig, ax = plt.subplots(1, 1, figsize=(12, 5))
    cmap = plt.cm.tab10
    for i, (name, curves) in enumerate(all_curves.items()):
        color = cmap(i)
        ax.plot(curves["step"], curves["train_loss"], label=f"{name} train loss", color=color)
        if curves.get("eval_loss"):
            ax.plot(curves["eval_step"], curves["eval_loss"], label=f"{name} eval loss",
                    color=color, linestyle="--", marker="o", markersize=2)
    ax.set_xlabel("Step")
    ax.set_ylabel("Loss")
    ax.set_title("Training Loss Comparison")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(str(OUT_DIR / "training_curves_comparison.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {OUT_DIR / 'training_curves_comparison.png'}")
else:
    print("No training curves available to plot.")

## Save Final Comparison

In [ ]:
# Save final comparison JSON with both EM and LLM judge metrics
final_comparison = {}
for name in judge_results:
    final_comparison[name] = {
        "em_metrics": em_results.get(name, {}),
        "judge_metrics": judge_results[name],
    }

comparison_path = OUT_DIR / "comparison_with_judge.json"
comparison_path.write_text(json.dumps(final_comparison, indent=2), encoding="utf-8")
print(f"Saved final comparison: {comparison_path}")
print(json.dumps(final_comparison, indent=2))